In [1]:
from dotenv import load_dotenv
load_dotenv()

from app.store import get_league_store
from app.models import LeagueModel

store = get_league_store()

print(LeagueModel.from_dict(store.get_by_name("欧冠杯")))
print(LeagueModel.from_dict(store.get_by_name("西甲"))) 


league_id='76' league_name='欧冠杯' cup='1'
league_id='26' league_name='西甲' cup=None


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from app.core.api_service import request_league_list
from app.store import get_league_store

import json
import requests
from bs4 import BeautifulSoup
import json
import os

# league_list = request_league_list()
# store = get_league_store()
# store.save_league_data(league_list)
# europe_leagues = store.get_by_continent("杯赛")
# print([l for l in europe_leagues])
url = os.getenv("OUHE_HTML_URL") + "league-center"
response = requests.get(url, proxies={"http": None}, timeout=5)
html = response.text
soup = BeautifulSoup(html, "html.parser")

script_tag = soup.find("script", id="__NEXT_DATA__", type="application/json")
json_object = json.loads(script_tag.get_text())
data = json_object["props"]["pageProps"]

print(json.dumps(data, indent=4, ensure_ascii=False))
# print(store.get_by_name("欧冠杯"))
# print(store.get_by_name("西甲"))








In [18]:
from dotenv import load_dotenv
load_dotenv()

import requests
from bs4 import BeautifulSoup
import json
import os
# from app.core.api_service import request_league_list
# from app.store import get_league_store


url = os.getenv("OUHE_HTML_URL") + "league-center" + "/detail?leagueId=51"
response = requests.get(url, proxies={"http": None}, timeout=5)
html = response.text
soup = BeautifulSoup(html, "html.parser")

script_tag = soup.find("script", id="__NEXT_DATA__", type="application/json")
json_object = json.loads(script_tag.get_text())
result = json_object["props"]["pageProps"]

print(result["seasonList"])
# print(result["leagueMatchRound"])
# print(result["data"])
print([(r["footballLeagueSubName"], r["footballLeagueSubId"]) for r in result["leagueMatchRound"]["footballLeagueSubArr"]])

# print(json.dumps(data, indent=4, ensure_ascii=False))

['2019-2021', '2023-2024']
[('外围赛', 2582), ('附加赛', 3250), ('附加赛决赛', 5123), ('分组赛', 3259), ('十六强', 3385), ('半准决赛', 3382), ('准决赛', 3383), ('决赛', 3384)]


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import requests
from bs4 import BeautifulSoup
import json
import os
from app.core.api_service import client
from app.store import get_league_store
from app.models import LeagueModel



def get_league_detail(league: LeagueModel) -> dict:
    url = f"{os.getenv("OUHE_HTML_URL")}league-center/detail?leagueId={league.league_id}"
    response = requests.get(url, proxies={"http": None}, timeout=5)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")

    script_tag = soup.find("script", id="__NEXT_DATA__", type="application/json")
    json_object = json.loads(script_tag.get_text())
    result = json_object["props"]["pageProps"]

    for season in result["seasonList"]:
        print(f"league：{league.league_name}， season: {season}")
        res = client.post("/web/leagueSummaryWeb", json={"channel": "web", "os": "browser", "leagueId": league.league_id, "season": season,})
        if league.is_cup:
            tmp_list = res["data"]["schedule"]["footballLeagueSubArr"]
            for t in tmp_list:
                print(f"阶段:{t["footballLeagueSubName"]}, 阶段ID:{t["footballLeagueSubId"]}")
                if t["isCurrentSclass"]:
                    print(f"当前阶段：{t["footballLeagueSubName"]}")
        else:
            print(f"一共 {res['data']['summary']["totalRound"]} 轮比赛")
            print(f"当前轮次：{res['data']['summary']["round"]}")

        # 杯赛通过 schedule.footballLeagueSubArr 获取轮次
        # 杯赛通过 schedule.footballLeagueSubArr.isCurrentSclass 判断杯赛当前的轮次
        #  联赛的话通过 round 获取当前轮次 


store = get_league_store()

league = LeagueModel.from_dict(store.get_by_name("欧冠杯"))
get_league_detail(league)

league：欧冠杯， season: 2024-2025
阶段:第一圈, 阶段ID:1972
阶段:第二圈, 阶段ID:2051
阶段:第三圈, 阶段ID:2102
阶段:附加赛, 阶段ID:2163
阶段:联赛阶段, 阶段ID:5435
阶段:淘汰赛, 阶段ID:5678
阶段:十六强, 阶段ID:2620
阶段:半准决赛, 阶段ID:2826
阶段:准决赛, 阶段ID:2913
阶段:决赛, 阶段ID:2965
当前阶段：决赛
league：欧冠杯， season: 2025-2026
阶段:第一圈, 阶段ID:1972
阶段:第二圈, 阶段ID:2051
阶段:第三圈, 阶段ID:2102
阶段:附加赛, 阶段ID:2163
阶段:联赛阶段, 阶段ID:5435
当前阶段：联赛阶段


In [ ]:
league = LeagueModel.from_dict(store.get_by_name("英超"))
get_league_detail(league)

league：英超， season: 2024-2025
一共 38 轮比赛
当前轮次：38
league：英超， season: 2025-2026
一共 38 轮比赛
当前轮次：12
